In [ ]:
# Cell 1: Import libraries and keyword groups

import json
import requests

from keywords import ARCHITECTURE_KEYWORDS, IT_KEYWORDS, ROLE_KEYWORDS

In [2]:
# Cell 2: Parent and child classes

class Job:
    def __init__(self, keywords):
        self.keywords = keywords

    def get_category(self):
        return "okänd"

    def __str__(self):
        return self.get_category()


class ArchitectureJob(Job):
    def get_category(self):
        return "bygg/arkitektur"

In [3]:
# Cell 3: Classify construction and architecture searches

def is_architect_term(word):
    cleaned_word = word.strip().lower()
    return cleaned_word in ROLE_KEYWORDS


def ad_contains_all_words(ad_text, search_words):
    text = ad_text.lower()
    for word in search_words:
        if word.strip().lower() not in text:
            return False
    return True


def classify_job_text(text):
    # Treat hyphenated terms such as IT-arkitekt as separate words.
    words = text.lower().replace("-", " ").split()

    has_it_keyword = any(word in IT_KEYWORDS for word in words)
    has_architecture_keyword = any(
        word in ARCHITECTURE_KEYWORDS or word in ROLE_KEYWORDS
        for word in words
    )

    if has_architecture_keyword and not has_it_keyword:
        return "bygg/arkitektur"

    return "okänd"


def classify_search_words(words):
    cleaned_words = [word.strip().lower() for word in words]

    has_architecture_keyword = any(
        word in ARCHITECTURE_KEYWORDS or word in ROLE_KEYWORDS
        for word in cleaned_words
    )

    if has_architecture_keyword:
        return ArchitectureJob(cleaned_words)

    return Job(cleaned_words)

In [4]:
# Cell 4: Fetch and clean API data

API_URL = "https://jobsearch.api.jobtechdev.se/search"


def fetch_jobs(search_words, limit=10):
    search_text = " ".join(search_words)

    parameters = {
        "q": search_text,
        "limit": limit,
    }

    try:
        response = requests.get(API_URL, params=parameters, timeout=10)
        response.raise_for_status()
        result = response.json()
        return result["hits"]
    except requests.exceptions.RequestException:
        print("Kunde inte hämta jobb från API:et.")
        return []
    except (ValueError, KeyError):
        print("API:et returnerade ett oväntat svar.")
        return []


def clean_job(job):
    description = job.get("description", {})
    employer = job.get("employer", {})
    workplace = job.get("workplace_address", {})
    occupation = job.get("occupation", {})
    employment_type = job.get("employment_type", {})
    working_hours = job.get("working_hours_type", {})

    return {
        "id": job.get("id", ""),
        "webpage_url": job.get("webpage_url", ""),
        "headline": job.get("headline", ""),
        "short_description": description.get("needs", ""),
        "description": description.get("text", ""),
        "employer": employer.get("name", ""),
        "city": workplace.get("city", ""),
        "municipality": workplace.get("municipality", ""),
        "occupation": occupation.get("label", ""),
        "employment_type": employment_type.get("label", ""),
        "working_hours": working_hours.get("label", ""),
        "application_deadline": job.get("application_deadline", ""),
        "number_of_vacancies": job.get("number_of_vacancies", ""),
    }

In [5]:
# Cell 5: Build and print the final output

def build_final_output(cleaned_jobs, answer_jobs, selected_field):
    return {
        "search_words": answer_jobs,
        "field": str(selected_field),
        "jobs": [
            {
                "headline": job.get("headline", ""),
                "employer": job.get("employer", ""),
                "city": job.get("city", ""),
                "category": str(selected_field),
                "webpage_url": job.get("webpage_url", ""),
            }
            for job in cleaned_jobs
        ],
    }


def print_result(result):
    print("Search words:", result["search_words"])
    print("Selected field:", result["field"])

    for i, job in enumerate(result["jobs"], start=1):
        print(f"\n{i}. {job['headline']}")
        print(f"   Employer: {job['employer']}")
        print(f"   City: {job['city']}")
        print(f"   Category: {job['category']}")

In [6]:
# Cell 6: Main construction-job search workflow

def main():
    # Read and validate one or two search keywords.
    try:
        answer_job = input(
            "Välj arkitektroll eller ett keyword: "
        )
        answer_jobs = answer_job.strip().lower().split()

        if len(answer_jobs) == 0:
            raise ValueError("Vänligen ange minst ett nyckelord.")

        if len(answer_jobs) > 2:
            raise ValueError("Vänligen ange endast ett eller två nyckelord.")

    except (EOFError, KeyboardInterrupt):
        print("Inmatningen avbröts.")
        return
    except ValueError as error:
        print(error)
        return

    # Only architecture and construction-related searches are supported.
    selected_field = classify_search_words(answer_jobs)

    if str(selected_field) != "bygg/arkitektur":
        print(
            "Ange arkitekt, architecture, arkitektur, architect "
            "eller ett bygg-/projekteringsnyckelord."
        )
        return

    # Fetch and clean a limited number of jobs for the prototype.
    jobs = fetch_jobs(answer_jobs, limit=5)

    if not jobs:
        print("Inga jobb hittades.")
        return

    cleaned_jobs = [clean_job(job) for job in jobs]
    filtered_jobs = []

    # Keep only jobs matching the search words and construction field.
    for job in cleaned_jobs:
        text = (
            job.get("headline", "") + " " +
            job.get("description", "") + " " +
            job.get("employer", "") + " " +
            job.get("occupation", "")
        ).lower()

        job_field = classify_job_text(text)

        if (
            ad_contains_all_words(text, answer_jobs)
            and job_field == str(selected_field)
        ):
            filtered_jobs.append(job)

    print("Jobs after keyword filtering:", len(filtered_jobs))

    result = build_final_output(
        filtered_jobs,
        answer_jobs,
        selected_field
    )

    print_result(result)

In [7]:
# Cell 7: Run the program

main()

Jobs after keyword filtering: 4
Search words: ['bygglov']
Selected field: bygg/arkitektur

1. Bygglovvägledare - stadsbyggnadsförvaltningens område för bygglov
   Employer: Uppsala kommun
   City: UPPSALA
   Category: bygg/arkitektur

2. Exploateringsingenjör
   Employer: Hässleholms kommun
   City: Hässleholm
   Category: bygg/arkitektur

3. Bygglovshandläggare/byggnadsinspektör
   Employer: Lycksele kommun
   City: Lycksele
   Category: bygg/arkitektur

4. Byggnadsinspektör/Bostadsanpassningshandläggare
   Employer: Emmaboda kommun
   City: Emmaboda
   Category: bygg/arkitektur
